# Bài thực hành: Khai thác mẫu trong event log

## Mục tiêu
Bài này thực hiện khai thác tập phổ biến trên dữ liệu nhật ký sự kiện để tìm các tổ hợp sự kiện thường xuất hiện cùng nhau.

## Phạm vi
- Sử dụng chính chương trình của nhóm qua CLI tại `src/main.jl`.
- Không dùng thư viện Frequent Itemset Mining bên ngoài.
- Tập trung vào yêu cầu ứng dụng ở Chương 5(b) của đề CSC14004 P2.

In [ ]:
from pathlib import Path
import subprocess

def find_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / 'src' / 'main.jl').exists():
            return p
    raise FileNotFoundError('Khong tim thay thu muc du an chua src/main.jl')

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
SRC_DIR = PROJECT_ROOT / 'src'
NOTEBOOK_DIR = PROJECT_ROOT / 'notebooks'
DATA_DIR = NOTEBOOK_DIR / 'data'
OUTPUT_DIR = NOTEBOOK_DIR / 'output_eventlog'
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT =', PROJECT_ROOT)
print('SRC_DIR      =', SRC_DIR)
print('DATA_DIR     =', DATA_DIR)
print('OUTPUT_DIR   =', OUTPUT_DIR)

In [ ]:
# Event dictionary (ma hoa su kien -> ID nguyen)
event_map = {
    1:  'AUTH_SUCCESS',
    2:  'AUTH_FAILURE',
    3:  'PRIV_ESCALATION_ATTEMPT',
    4:  'MULTIPLE_404',
    5:  'SUDO_COMMAND',
    6:  'FILE_PERMISSION_DENIED',
    7:  'SERVICE_RESTART',
    8:  'OUTBOUND_CONN_NEW_IP',
    9:  'HIGH_CPU',
    10: 'BACKUP_COMPLETED',
    11: 'CONFIG_CHANGE',
    12: 'DATABASE_ERROR',
}

# Moi transaction la mot cua so thoi gian (vi du 5 phut) gom cac su kien xay ra cung luc.
transactions = [
    [1, 10],
    [1, 10, 7],
    [2, 3, 5, 6],
    [2, 3, 5, 8, 11],
    [1, 10, 9],
    [2, 3, 5, 6, 8],
    [7, 9, 12],
    [2, 3, 5, 11],
    [1, 10, 11],
    [2, 3, 5, 6],
    [7, 9, 12, 6],
    [2, 3, 5, 8],
    [1, 10],
    [2, 3, 5, 6, 11],
    [7, 9, 12],
    [1, 10, 4],
    [2, 3, 5, 6, 8, 11],
    [1, 10, 7],
    [2, 3, 5, 6],
    [7, 9, 12, 11],
]

input_file = DATA_DIR / 'event_log_demo.txt'
with open(input_file, 'w', encoding='utf-8') as f:
    for tr in transactions:
        f.write(' '.join(str(x) for x in sorted(set(tr))) + '\n')

map_file = DATA_DIR / 'event_id_mapping.txt'
with open(map_file, 'w', encoding='utf-8') as f:
    for k in sorted(event_map):
        f.write(f"{k}: {event_map[k]}\n")

print('Da tao input:', input_file)
print('Da tao mapping:', map_file)
print('So transaction:', len(transactions))

## Quy trình chạy thực nghiệm bằng CLI

### Bước 1. Chuẩn bị tham số
- Chọn `JULIA_CMD` phù hợp với máy đang dùng.
- Đặt `MINSUP` theo tỷ lệ trong khoảng `[0, 1]`.

### Bước 2. Chạy các chế độ
- `-a`: chạy từng thuật toán (`classic`, `projection`, `adjacency`).
- `-c`: so sánh hai thuật toán trên cùng input.
- `-ca`: chạy toàn bộ thuật toán trên cùng input.

### Lưu ý kỹ thuật
Implementation hiện tại diễn giải `minsup` theo tỷ lệ, ví dụ `0.30` nghĩa là ngưỡng hỗ trợ bằng 30% số transaction.

Nếu máy không nhận lệnh `julia`, cần đổi `JULIA_CMD` thành đường dẫn đầy đủ đến `julia.exe`.

In [ ]:
JULIA_CMD = 'julia'
MINSUP = 0.30

def run_cli(args):
    cmd = [JULIA_CMD, 'main.jl', *args]
    print('> ' + ' '.join(cmd))
    try:
        result = subprocess.run(
            cmd,
            cwd=SRC_DIR,
            text=True,
            capture_output=True,
            check=False,
        )
    except FileNotFoundError as e:
        raise RuntimeError(
            'Khong tim thay Julia. Hay sua JULIA_CMD = duong dan den julia.exe, vi du C:/Users/.../Julia-1.x/bin/julia.exe'
        ) from e

    if result.stdout.strip():
        print(result.stdout)
    if result.stderr.strip():
        print('STDERR:')
        print(result.stderr)

    if result.returncode != 0:
        raise RuntimeError(f'CLI failed with code {result.returncode}')

# -a: chay tung thuat toan
for alg in ['classic', 'projection', 'adjacency']:
    run_cli(['-a', alg, str(input_file), str(OUTPUT_DIR), str(MINSUP)])

# -c: so sanh 2 thuat toan
run_cli(['-c', 'classic', 'projection', str(input_file), str(OUTPUT_DIR), str(MINSUP)])

# -ca: chay tat ca
run_cli(['-ca', str(input_file), str(OUTPUT_DIR), str(MINSUP)])

In [ ]:
def minsup_label(minsup: float) -> str:
    percent = round(minsup * 100, 2)
    if float(percent).is_integer():
        return f"{int(percent)}%"
    return f"{percent}%"

def parse_output(path: Path):
    patterns = []
    for line in path.read_text(encoding='utf-8').splitlines():
        line = line.strip()
        if not line:
            continue
        left, right = line.split('#SUP:')
        items = [int(x) for x in left.strip().split()] if left.strip() else []
        sup = int(right.strip())
        patterns.append({'items': items, 'support': sup})
    return patterns

def decode_pattern(items):
    return [event_map[i] for i in items]

label = minsup_label(MINSUP)
base = input_file.stem

all_patterns = {}
for alg in ['classic', 'projection', 'adjacency']:
    out_file = OUTPUT_DIR / f'local_{alg}_{base}_{label}.txt'
    if not out_file.exists():
        print(f'[WARN] Chua co file output: {out_file}')
        continue
    patterns = parse_output(out_file)
    all_patterns[alg] = patterns

    print('\n' + '=' * 70)
    print(f'Algorithm: {alg} | so pattern: {len(patterns)}')
    print('=' * 70)

    for p in sorted(patterns, key=lambda x: (-x['support'], -len(x['items']), x['items']))[:12]:
        print(f"support={p['support']:>2} | ids={p['items']} | events={decode_pattern(p['items'])}")

In [ ]:
# Danh sach su kien co tinh chat canh bao
suspicious_ids = {2, 3, 5, 6, 8, 11, 12}

classic_patterns = all_patterns.get('classic', [])
suspicious_patterns = []
for p in classic_patterns:
    overlap = sorted(set(p['items']) & suspicious_ids)
    if len(overlap) >= 2:
        suspicious_patterns.append((p, overlap))

suspicious_patterns.sort(key=lambda x: (-x[0]['support'], -len(x[0]['items']), x[0]['items']))

print('Top suspicious frequent combinations (classic):')
for p, overlap in suspicious_patterns[:10]:
    print(
        f"support={p['support']:>2} | pattern={decode_pattern(p['items'])} | suspicious_part={decode_pattern(overlap)}"
    )

print('\nInterpretation:')
print('- Neu combo AUTH_FAILURE + PRIV_ESCALATION_ATTEMPT + SUDO_COMMAND xuat hien lap lai, co the la mau tan cong leo thang dac quyen.')
print('- Neu FILE_PERMISSION_DENIED / OUTBOUND_CONN_NEW_IP / CONFIG_CHANGE di cung cac su kien tren, can uu tien dieu tra.')
print('- Frequent itemset giup tim mau lap lai; de bat anomaly hiem, can giam minsup hoac ket hop detector theo thoi gian.')

## Nhận xét kết quả và ý nghĩa ứng dụng

### 1. Mẫu đồng xuất hiện trong log
Frequent itemset cho thấy những nhóm sự kiện lặp lại nhiều lần trong hệ thống. Đây là cơ sở để mô tả hành vi vận hành bình thường hoặc hành vi cần theo dõi.

### 2. Khả năng phát hiện lỗi và rủi ro bảo mật
Các mẫu có support cao chứa những sự kiện như `AUTH_FAILURE`, `PRIV_ESCALATION_ATTEMPT`, `SUDO_COMMAND` có thể phản ánh chuỗi hành vi tấn công hoặc cấu hình sai lặp lại.

### 3. Cách dùng trong giám sát thực tế
Có thể dùng các mẫu frequent làm baseline. Khi một pattern có support tăng mạnh theo thời gian, hệ thống có thể phát cảnh báo sớm để kiểm tra.

### 4. Hạn chế
Anomaly hiếm thường không đủ support để trở thành frequent itemset. Vì vậy cần kết hợp thêm các kỹ thuật khác như giảm `minsup`, phân tích theo cửa sổ thời gian, hoặc dùng thêm luật kết hợp (confidence/lift).

Phần thực hành này đáp ứng yêu cầu ứng dụng Chương 5(b): khai thác tổ hợp sự kiện thường xuyên trên event log và thảo luận khả năng phát hiện lỗi/hành vi bất thường.